Cloud-only data dependency: this notebook expects access to OncDRS/cloud data paths and will not run in a local-only environment.

# Binary NEPC: Generate Notes and Run the LLM

This notebook uses the same raw OncDRS note-reading and cohort-MRN source logic as the pipeline scripts.

Workflow:
1. Reuse existing compiled artifacts when available.
2. Otherwise infer or specify the MRN cohort.
3. Read raw OncDRS notes.
4. Optionally write a compiled notes CSV and/or gzip bundle.
5. Build trigger-centered snippets only when no snippet bundle already exists.
6. Run the NEPC classifier on one patient.

In [ ]:
from pathlib import Path
import gzip
import json
import sys

import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "binary_NEPC" else Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from binary_NEPC.run_NEPC_classifier import classify_patient
from shared.compile_prostate_notes import DEFAULT_PROSTATE_MRN_SOURCE, derive_prostate_mrns
from shared.llm_helpers import (
    DEFAULT_OUTPUT_DIR,
    DEFAULT_RAW_TEXT_PATHS,
    NOTE_BUNDLE_FILENAME,
    PROSTATE_TEXT_CSV,
    build_client,
    build_patient_snippets,
    load_notes_csv,
    load_raw_text_notes,
    parse_mrn_values,
    resolve_raw_text_paths,
    write_note_bundle,
    write_notes_csv,
)

pd.set_option("display.max_colwidth", 200)


def load_snippet_bundle(path):
    with gzip.open(path, "rt", encoding="utf-8") as handle:
        payload = json.load(handle)
    snippets = payload.get("patient_snippets", {})
    return {int(mrn): rows for mrn, rows in snippets.items()}


def write_snippet_bundle(path, patient_snippets):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "patient_snippets": {str(mrn): rows for mrn, rows in patient_snippets.items()},
    }
    with gzip.open(path, "wt", encoding="utf-8") as handle:
        json.dump(payload, handle, ensure_ascii=False)


In [ ]:
# Parameters
# Leave MRNS empty to infer the full cohort from the default MRN source.
MRNS = []
MRN_FILE = None

# Uses the default prostate MRN cohort file by default.
COHORT_SOURCE = DEFAULT_PROSTATE_MRN_SOURCE

# Defaults to the shared raw note locations from shared.llm_helpers.
RAW_TEXT_PATHS = list(DEFAULT_RAW_TEXT_PATHS)

# Output locations for optional compiled artifacts.
NOTES_CSV_PATH = PROSTATE_TEXT_CSV
NOTE_BUNDLE_PATH = DEFAULT_OUTPUT_DIR / NOTE_BUNDLE_FILENAME
SNIPPET_BUNDLE_PATH = DEFAULT_OUTPUT_DIR / "LLM_NEPC_classifier_patient_snippets.json.gz"
WRITE_NOTES_CSV = False
WRITE_NOTE_BUNDLE = False
WRITE_SNIPPET_BUNDLE = False

# Snippet / classification controls
MAX_NOTES_PER_PATIENT = 30
MODEL_NAME = "gpt-4o"
MAX_RETRIES = 3

# Optional: set this to a specific MRN to inspect and classify.
TARGET_MRN = None

In [ ]:
notes_csv_exists = Path(NOTES_CSV_PATH).exists()
note_bundle_exists = Path(NOTE_BUNDLE_PATH).exists()
snippet_bundle_exists = Path(SNIPPET_BUNDLE_PATH).exists()

print(f"Compiled prostate_text_data.csv exists: {notes_csv_exists}")
print(f"Path: {NOTES_CSV_PATH}")
print(f"Existing binary NEPC note bundle exists: {note_bundle_exists}")
print(f"Path: {NOTE_BUNDLE_PATH}")
print(f"Existing snippet bundle exists: {snippet_bundle_exists}")
print(f"Path: {SNIPPET_BUNDLE_PATH}")

In [ ]:
selected_mrns = set()

if MRNS:
    selected_mrns |= parse_mrn_values(MRNS)

if MRN_FILE is not None:
    mrn_file_path = Path(MRN_FILE)
    if not mrn_file_path.exists():
        raise FileNotFoundError(f"MRN file not found: {mrn_file_path}")
    file_mrns = pd.read_csv(mrn_file_path, header=None).iloc[:, 0].tolist()
    selected_mrns |= parse_mrn_values(file_mrns)

raw_text_paths = resolve_raw_text_paths(RAW_TEXT_PATHS)
explicit_mrn_selection = bool(selected_mrns)

if notes_csv_exists and not explicit_mrn_selection:
    selected_mrns = None
    print("Using existing compiled notes CSV; skipping cohort-source MRN derivation.")
elif not selected_mrns:
    selected_mrns = derive_prostate_mrns(COHORT_SOURCE)
    print(f"Derived cohort from cohort source: {len(selected_mrns)} MRNs")
else:
    print(f"Using explicit MRN selection: {len(selected_mrns)} MRNs")

print("Cohort source:", COHORT_SOURCE)
print("Raw note roots:")
for path in raw_text_paths:
    print("  ", path)

In [ ]:
if notes_csv_exists:
    notes_df = load_notes_csv(NOTES_CSV_PATH, selected_mrns=selected_mrns)
    print(f"Loaded existing notes CSV: {NOTES_CSV_PATH}")
else:
    notes_df = load_raw_text_notes(raw_text_paths, selected_mrns)
    print("Loaded notes from raw OncDRS JSONs")

print(f"Loaded {len(notes_df):,} notes for {notes_df['DFCI_MRN'].nunique():,} patients")
display(notes_df.head())

In [ ]:
if WRITE_NOTES_CSV and not notes_csv_exists:
    standardized = write_notes_csv(NOTES_CSV_PATH, notes_df)
    print(f"Wrote notes CSV: {NOTES_CSV_PATH}")
    print(f"CSV rows: {len(standardized):,}")
elif WRITE_NOTES_CSV and notes_csv_exists:
    print(f"Skipped writing notes CSV because it already exists: {NOTES_CSV_PATH}")

if WRITE_NOTE_BUNDLE and not note_bundle_exists:
    write_note_bundle(
        NOTE_BUNDLE_PATH,
        notes_df,
        raw_text_paths=raw_text_paths,
        selected_mrns=selected_mrns,
    )
    print(f"Wrote note bundle: {NOTE_BUNDLE_PATH}")
elif WRITE_NOTE_BUNDLE and note_bundle_exists:
    print(f"Skipped writing note bundle because it already exists: {NOTE_BUNDLE_PATH}")

In [ ]:
if snippet_bundle_exists:
    patient_snippets = load_snippet_bundle(SNIPPET_BUNDLE_PATH)
    print(f"Loaded existing snippet bundle: {SNIPPET_BUNDLE_PATH}")
else:
    patient_snippets = build_patient_snippets(
        notes_df,
        max_notes_per_patient=MAX_NOTES_PER_PATIENT,
    )
    print("Built snippets from notes_df")
    if WRITE_SNIPPET_BUNDLE:
        write_snippet_bundle(SNIPPET_BUNDLE_PATH, patient_snippets)
        print(f"Wrote snippet bundle: {SNIPPET_BUNDLE_PATH}")

if selected_mrns is not None:
    selected_mrn_set = {int(mrn) for mrn in selected_mrns}
    patient_snippets = {mrn: rows for mrn, rows in patient_snippets.items() if int(mrn) in selected_mrn_set}

triggered_mrns = sorted(patient_snippets)
print(f"Patients with triggered snippets: {len(triggered_mrns):,}")

snippet_summary = pd.DataFrame(
    [
        {
            "DFCI_MRN": mrn,
            "num_snippets": len(snippets),
            "note_types": sorted({s['note_type'] for s in snippets}),
            "trigger_categories": sorted({c for s in snippets for c in s['trigger_categories']}),
        }
        for mrn, snippets in patient_snippets.items()
    ]
).sort_values(["num_snippets", "DFCI_MRN"], ascending=[False, True])

display(snippet_summary.head(20))

In [ ]:
if not triggered_mrns:
    raise ValueError("No trigger-bearing patients were found in the selected note set.")

mrn_to_review = int(TARGET_MRN) if TARGET_MRN is not None else int(triggered_mrns[0])
snippets = patient_snippets[mrn_to_review]

print(f"Reviewing MRN: {mrn_to_review}")
print(f"Snippets sent to model: {len(snippets)}")

snippet_df = pd.DataFrame(snippets)
display(snippet_df[["note_date", "note_type", "trigger_categories"]])
display(snippet_df[["note_date", "note_type", "snippet"]].head(5))

In [ ]:
payload_preview = {
    "patient_mrn": mrn_to_review,
    "notes": [
        {
            "note_date": s["note_date"],
            "note_type": s["note_type"],
            "trigger_categories": s["trigger_categories"],
            "note_text": s["snippet"],
        }
        for s in snippets
    ],
}

print(json.dumps(payload_preview, ensure_ascii=False, indent=2)[:6000])

In [ ]:
client = build_client()
result, error = classify_patient(
    client=client,
    model=MODEL_NAME,
    max_retries=MAX_RETRIES,
    mrn=mrn_to_review,
    snippets=snippets,
)

if error:
    raise RuntimeError(error)

result